In [ ]:
# basic
import numpy as np
import pandas as pd

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder

# pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# imbalance
from imblearn.over_sampling import SMOTE

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier

# evaluation
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

In [ ]:
train_df = pd.read_csv(r"C:\Users\Osama Youssef\Downloads\AI Projects\fraud-detection-system\data\fraudTrain.csv")
test_df = pd.read_csv(r"C:\Users\Osama Youssef\Downloads\AI Projects\fraud-detection-system\data\fraudTest.csv")

In [ ]:
print(train_df.shape)
print(test_df.shape)

In [ ]:
train_df.head()

In [ ]:
train_df.info(),train_df.describe()

In [ ]:
train_df = train_df.drop(columns =['Unnamed: 0','cc_num','first','last','street','trans_num','zip'])
test_df = test_df.drop(columns =['Unnamed: 0','cc_num','first','last','street','trans_num','zip'])

In [ ]:
## check outliers
def outliers(df,column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column]<lower_bound)|(df[column]>upper_bound)]
    return outliers
num_cols = train_df.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    out = outliers(train_df, col)
    percent = len(out) / len(train_df) * 100
    print(col, ":", round(percent,2), "%")

In [ ]:
## isolate time
train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])
train_df["hour"] = train_df["trans_date_trans_time"].dt.hour
train_df = train_df.drop(columns = 'trans_date_trans_time')

test_df["trans_date_trans_time"] = pd.to_datetime(test_df["trans_date_trans_time"])
test_df["hour"] = test_df["trans_date_trans_time"].dt.hour
test_df = test_df.drop(columns = 'trans_date_trans_time')

In [ ]:
##calculate age
train_df["dob"] = pd.to_datetime(train_df["dob"])
train_df["age"] = (pd.Timestamp.now() - train_df["dob"]).dt.days // 365
train_df = train_df.drop(columns = 'dob')

test_df["dob"] = pd.to_datetime(test_df["dob"])
test_df["age"] = (pd.Timestamp.now() - test_df["dob"]).dt.days // 365
test_df = test_df.drop(columns = 'dob')

In [ ]:
## calculate distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c
train_df["distance"] = haversine(
    train_df["lat"],
    train_df["long"],
    train_df["merch_lat"],
    train_df["merch_long"])
test_df["distance"] = haversine(
    test_df["lat"],
    test_df["long"],
    test_df["merch_lat"],
    test_df["merch_long"])
cols_to_drop = ["lat", "long", "merch_lat", "merch_long"]
train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
oo = OneHotEncoder(sparse_output=False)
encoded = oo.fit_transform(train_df[['gender','category','state']])
encoded_df = pd.DataFrame(
    encoded,
    columns=oo.get_feature_names_out(['gender','category','state'])
)
train_df = pd.concat([train_df.drop(['gender','category','state'], axis=1), encoded_df], axis=1)
onehot_cols = train_df.filter(regex='^(gender_|category_|state_)').columns
train_df[onehot_cols] = train_df[onehot_cols].astype('bool')

In [ ]:
encoded_test = oo.transform(test_df[['gender','category','state']])
encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=oo.get_feature_names_out(['gender','category','state'])
)
test_df = pd.concat(
    [test_df.drop(['gender','category','state'], axis=1), encoded_test_df],
    axis=1
)
test_df[onehot_cols] = test_df[onehot_cols].astype('bool')

In [ ]:
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_df[['merchant','city','job']] = enc.fit_transform(train_df[['merchant','city','job']])
test_df[['merchant','city','job']] = enc.transform(test_df[['merchant','city','job']])

In [ ]:
X = train_df.drop("is_fraud", axis=1)
y = train_df["is_fraud"]

In [ ]:
ss = StandardScaler()
X = ss.fit_transform(X)

In [ ]:
X_test = test_df.drop("is_fraud", axis=1)
y_test = test_df["is_fraud"]

X_test = ss.transform(X_test)

In [ ]:
X,y,X_test,y_test

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [ ]:
IF = IsolationForest(n_estimators = 200,
max_samples = 10000,
contamination = 0.006,
n_jobs = -1,
random_state = 42)
IF.fit(X_train)
y_train_pred = IF.predict(X_train)

In [ ]:
pd.value_counts(y_train_pred)

In [ ]:
train_scores = IF.decision_function(X_train)
val_scores   = IF.decision_function(X_val)
test_scores  = IF.decision_function(X_test)

In [ ]:
X_train = np.hstack([X_train, train_scores.reshape(-1,1)])
X_val   = np.hstack([X_val, val_scores.reshape(-1,1)])
X_test  = np.hstack([X_test, test_scores.reshape(-1,1)])

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [ ]:
class FraudNN(nn.Module):

    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

class Trainer:

    def __init__(self, model, optimizer, loss_fn, device):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device

    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0
        for x, y in dataloader:
            x = x.to(self.device)
            y = y.to(self.device)
            preds = self.model(x)
            loss = self.loss_fn(preds, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        return avg_loss

    def validate(self, dataloader):
        self.model.eval()
        with torch.no_grad():
            for x, y in dataloader:
                x = x.to(self.device)
                y = y.to(self.device)
                preds = self.model(x)

class FraudDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
config = {
    "data": {
        "batch_size": 512
    },

    "model": {
        "input_dim": 77,
        "hidden_dims": [128, 64],
        "dropout": 0.2
    },

    "optimizer": {
        "type": "adam",
        "lr": 0.001
    },

    "training": {
        "epochs": 20
    }
}

In [ ]:
train_dataset = FraudDataset(X_train_smote, y_train_smote)
val_dataset   = FraudDataset(X_val, y_val)
test_dataset  = FraudDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=config["data"]["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config["data"]["batch_size"])
test_loader = DataLoader(test_dataset, batch_size=config["data"]["batch_size"])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = FraudNN(**config["model"]).to(device)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["optimizer"]["lr"]
)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device="cuda"
)

In [ ]:
for epoch in range(config["training"]["epochs"]):
    trainer.train_epoch(train_loader)
    trainer.validate(val_loader)

trainer.validate(test_loader)

In [ ]:
train_losses = []
val_losses = []

val_precision = []
val_recall = []
val_f1 = []
val_auc = []

In [ ]:
def evaluate_model(model, dataloader):

    model.eval()

    all_preds = []
    all_probs = []
    all_targets = []

    total_loss = 0

    with torch.no_grad():

        for x, y in dataloader:

            x = x.to(device)
            y = y.to(device)

            preds = model(x)

            loss = loss_fn(preds, y)

            total_loss += loss.item()

            probs = torch.sigmoid(preds)

            predicted = (probs > 0.9).float()

            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(predicted.cpu().numpy().flatten())
            all_targets.extend(y.cpu().numpy().flatten())

    avg_loss = total_loss / len(dataloader)

    precision = precision_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds)
    auc = roc_auc_score(all_targets, all_probs)

    return avg_loss, precision, recall, f1, auc, all_probs, all_preds, all_targets

In [ ]:
for epoch in range(config["training"]["epochs"]):

    train_loss = trainer.train_epoch(train_loader)

    val_loss, precision, recall, f1, auc, probs, preds, targets = evaluate_model(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    val_precision.append(precision)
    val_recall.append(recall)
    val_f1.append(f1)
    val_auc.append(auc)

    print(f"\nEpoch {epoch+1}")

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")

    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"AUC: {auc:.4f}")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.title("Training vs Validation Loss")

plt.show()

In [ ]:
cm = confusion_matrix(targets, preds)

plt.imshow(cm)

plt.title("Confusion Matrix")

plt.colorbar()

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
fraud_indices = np.where(np.array(targets) == 1)[0]

for i in fraud_indices[:10]:
    print("Prediction:", preds[i], "Actual:", targets[i])

In [ ]:
print(torch.cuda.is_available())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
val_loss, precision, recall, f1, auc, probs, preds, targets = evaluate_model(model, val_loader)

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)

for t in thresholds:

    new_preds = (np.array(probs) > t).astype(int)

    precision = precision_score(targets, new_preds)
    recall = recall_score(targets, new_preds)
    f1 = f1_score(targets, new_preds)

    print(f"Threshold {t:.2f} | Precision {precision:.3f} | Recall {recall:.3f} | F1 {f1:.3f}")

In [ ]:
def rule_engine(transaction, fraud_prob, anomaly_score):
    risk_points = 0
    flags = []

    # القاعدة 1: المبالغ الكبيرة جداً (High Amount Rule)
    if transaction['amt'] > 5000:
        risk_points += 3
        flags.append("HIGH_AMOUNT")

    # القاعدة 2: معاملات الفجر (Night Owl Rule)
    if 2 <= transaction['hour'] <= 5 and transaction['amt'] > 500:
        risk_points += 2
        flags.append("NIGHT_TRANSACTION")

    # القاعدة 3: المسافات البعيدة جداً (Geographic Risk)
    if transaction['distance'] > 200:
        risk_points += 2
        flags.append("LARGE_DISTANCE")
        
    # دمج مخرجات الـ Anomaly Detection (Isolation Forest)
    if anomaly_score < -0.5: # القيم السالبة جداً تعني شذوذ قوي
        risk_points += 1
        flags.append("ANOMALOUS_BEHAVIOR")

    return risk_points, flags

In [ ]:
class DecisionLayer:

    def __init__(self,
                 fraud_threshold=0.9,
                 anomaly_threshold=-0.2,
                 rule_threshold=3):

        self.fraud_threshold = fraud_threshold
        self.anomaly_threshold = anomaly_threshold
        self.rule_threshold = rule_threshold

    def decide(self, fraud_prob, anomaly_score, rule_score):

        if fraud_prob > self.fraud_threshold:
            return "DECLINE"

        elif anomaly_score < self.anomaly_threshold:
            return "REVIEW"

        elif rule_score > self.rule_threshold:
            return "REVIEW"

        else:
            return "APPROVE"

In [ ]:
class FraudSystem:

    def __init__(self, model, anomaly_model):

        self.model = model
        self.anomaly_model = anomaly_model

        self.rules = RuleEngine()
        self.decision = DecisionLayer()

    def predict(self, transaction_features, raw_transaction):

        x = torch.tensor(transaction_features, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = self.model(x)
            prob = torch.sigmoid(logits).item()

        anomaly = self.anomaly_model.decision_function([transaction_features])[0]

        rule_score, flags = self.rules.evaluate(raw_transaction)

        decision = self.decision.decide(prob, anomaly, rule_score)

        return {
            "fraud_probability": prob,
            "anomaly_score": anomaly,
            "rule_score": rule_score,
            "flags": flags,
            "decision": decision
        }

# gg